# Customer Churn Prediction — Iranian Telecom Dataset

This notebook carries out the full process described in `instruct.md` and `PLAN.md`: data cleaning, exploratory analysis, preprocessing, model comparison, threshold selection, risk scoring, feature importance, and business recommendations — end to end, with the reasoning kept next to the code that implements it.

## 1. Setup & raw load

Imports and a first look at the raw file before touching anything.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
plt.rcParams["figure.dpi"] = 100

RAW_PATH = Path("Customer Churn.csv")
DATA_DIR = Path("data")
MODELS_DIR = Path("models")
OUTPUTS_DIR = Path("outputs")
for d in (DATA_DIR, MODELS_DIR, OUTPUTS_DIR):
    d.mkdir(exist_ok=True)

raw = pd.read_csv(RAW_PATH)
print("Shape:", raw.shape)
raw.head()

Shape: (3150, 14)


,Call Failure,Complains,Subscription Length,Charge Amount,Seconds of Use,Frequency of use,Frequency of SMS,Distinct Called Numbers,Age Group,Tariff Plan,Status,Age,Customer Value,Churn
0,8,0,38,0,4370,71,5,17,3,1,1,30,197.640,0
1,0,0,39,0,318,5,7,4,2,1,2,25,46.035,0
2,10,0,37,0,2453,60,359,24,3,1,1,30,1536.520,0
3,10,0,38,0,4198,66,1,35,1,1,1,15,240.020,0
4,3,0,38,0,2393,58,2,33,1,1,1,15,145.805,0


## 2. Data cleaning

Findings from the initial exploration that drive the cleaning decisions below:

- **300 exact duplicate rows** (~9.5% of the data) — dropped outright, they add no information and would bias training.
- **`Age` is redundant** — every row's `Age` is fully determined by `Age Group` (1→15, 2→25, 3→30, 4→45, 5→55). It's a bucket label dressed up as a number, not a real age. We keep `Age Group` (honestly ordinal) and drop `Age`.
- **Column names have inconsistent double spaces** (`'Call  Failure'`, `'Subscription  Length'`, `'Charge  Amount'`) — normalized to clean snake_case.
- **132 customers (4.2%) are fully dormant**: zero `Seconds of Use`, zero `Frequency of use`, zero `Frequency of SMS`, and `Customer Value` = 0. We don't drop them — a `is_dormant` flag is added instead, since dropping would throw away real customers and their churn behavior is itself informative (52% churn rate among them, vs 15.7% overall).
- No missing values anywhere in the raw file.

In [2]:
rename_map = {
    "Call  Failure": "call_failure",
    "Complains": "complains",
    "Subscription  Length": "subscription_length",
    "Charge  Amount": "charge_amount",
    "Seconds of Use": "seconds_of_use",
    "Frequency of use": "frequency_of_use",
    "Frequency of SMS": "frequency_of_sms",
    "Distinct Called Numbers": "distinct_called_numbers",
    "Age Group": "age_group",
    "Tariff Plan": "tariff_plan",
    "Status": "status",
    "Age": "age",
    "Customer Value": "customer_value",
    "Churn": "churn",
}
df = raw.rename(columns=rename_map)

n_before = len(df)
n_dupes = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)

# Age is fully determined by age_group -- verify before dropping
assert df.groupby("age_group")["age"].nunique().max() == 1, \
    "age is no longer a 1:1 function of age_group -- investigate before dropping"
df = df.drop(columns=["age"])

df["is_dormant"] = (
    (df["seconds_of_use"] == 0)
    & (df["frequency_of_use"] == 0)
    & (df["frequency_of_sms"] == 0)
    & (df["customer_value"] == 0)
).astype(int)

df.to_csv(DATA_DIR / "customer_churn_cleaned.csv", index=False)

print(f"Rows before: {n_before}")
print(f"Exact duplicates removed: {n_dupes}")
print(f"Rows after: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"Dormant customers flagged: {df['is_dormant'].sum()} ({df['is_dormant'].mean():.1%})")
print(f"Churn rate after cleaning: {df['churn'].mean():.1%}")
df.head()

Rows before: 3150
Exact duplicates removed: 300
Rows after: 2850
Columns: ['call_failure', 'complains', 'subscription_length', 'charge_amount', 'seconds_of_use', 'frequency_of_use', 'frequency_of_sms', 'distinct_called_numbers', 'age_group', 'tariff_plan', 'status', 'customer_value', 'churn', 'is_dormant']
Dormant customers flagged: 85 (3.0%)
Churn rate after cleaning: 15.6%


,call_failure,complains,subscription_length,charge_amount,seconds_of_use,frequency_of_use,frequency_of_sms,distinct_called_numbers,age_group,tariff_plan,status,customer_value,churn,is_dormant
0,8,0,38,0,4370,71,5,17,3,1,1,197.640,0,0
1,0,0,39,0,318,5,7,4,2,1,2,46.035,0,0
2,10,0,37,0,2453,60,359,24,3,1,1,1536.520,0,0
3,10,0,38,0,4198,66,1,35,1,1,1,240.020,0,0
4,3,0,38,0,2393,58,2,33,1,1,1,145.805,0,0
